# Exercise 1: Web Scraping, Portfolio Analysis & Visualization

## Part 1: Web Scraping Top Stock Gainers from Yahoo Finance

- First, we'll set up our environment and scrape the necessary data from Yahoo Finance

In [2]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


- The opts configuration ensure the browser runs headlessly using disabling automation flags ans setting a realistic user-agent. This help to bypass bot detection mechanism in Yahoo Finance.
- Additionally, since we need to load many web pages to obtain all stocks possible. One variable set to change of url direction to scrape more information

In [ ]:
# Import necessary libraries
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, StaleElementReferenceException

# --- Step 1: Configure a Stealthy WebDriver ---
# This is the most critical fix. We configure the driver to avoid bot detection.
opts = Options()
# The following arguments help the driver appear as a normal browser
opts.add_argument("--headless=new") 
opts.add_argument("--start-maximized")
opts.add_argument("--disable-blink-features=AutomationControlled")
opts.add_experimental_option("excludeSwitches", ["enable-automation"])
opts.add_experimental_option('useAutomationExtension', False)
opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")

# Initialize the configured driver
driver = webdriver.Chrome(options=opts)
print("WebDriver initialized with anti-detection settings.")

# --- Step 2: Scrape Data Using Direct URL Pagination ---
base_url = "https://finance.yahoo.com/markets/stocks/gainers"
all_stocks = set() # Using a set to automatically handle any duplicates

# Yahoo uses 'offset' to control the starting point of the list.
for offset in [0, 25, 50, 75]: # Scrape first 4 pages to ensure we get 50+ unique stocks
    
    # Construct the URL for the specific page
    target_url = f"{base_url}?start0=&count=25&start={offset}"
    print(f"Navigating to: {target_url}")
    
    try:
        driver.get(target_url)
        # Handle the cookie consent banner with a robust method
        try:
            # Wait for the cookie button to be clickable and click it
            cookie_button_xpath = "//button[contains(., 'Accept') or contains(., 'Agree')]"
            wait = WebDriverWait(driver, 10)
            accept_btn = wait.until(EC.element_to_be_clickable((By.XPATH, cookie_button_xpath)))
            accept_btn.click()
            print("Cookie consent banner handled.")
            time.sleep(1) # Give page time to adjust after banner closes
        except TimeoutException:
            print("Cookie consent banner not found or already handled.")

        # Wait for the table body to be present, which confirms data has loaded
        wait = WebDriverWait(driver, 20)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr")))
        
        # Extract data from the table rows
        rows = driver.find_elements(By.CSS_SELECTOR, "table tbody tr")
        
        for row in rows:
            try:
                # Get all cells in the row
                tds = row.find_elements(By.TAG_NAME, "td")
                if len(tds) >= 2: # Ensure the row has at least symbol and name
                    symbol = tds[0].text.strip()
                    name = tds[1].text.strip()
                    if symbol and name: # Add only if both are non-empty
                        all_stocks.add((symbol, name))
            except StaleElementReferenceException:
                # We simply skip that row and continue.
                continue

        #In case we are looking for exactly 50 stocks, we can break early     
        if len(all_stocks) >= 50:
            break
            
    except TimeoutException:
        print(f"Timed out waiting for content on page with offset {offset}. Moving on.")
        continue
    except Exception as e:
        print(f"An error occurred on offset {offset}: {e}")
        break

# Quit the driver
driver.quit()

# --- Step 3: Storage ---
final_stock_list = list(all_stocks)[:50]
gainers_df = pd.DataFrame(final_stock_list, columns=['Symbol', 'Name'])

# Display the final DataFrame
print("\n Scraping Completed Stock Gainers")
print(gainers_df)

WebDriver initialized with anti-detection settings.
Navigating to: https://finance.yahoo.com/markets/stocks/gainers?start0=&count=25&start=0
Cookie consent banner not found or already handled.
Navigating to: https://finance.yahoo.com/markets/stocks/gainers?start0=&count=25&start=25
Cookie consent banner not found or already handled.

 Scraping Completed Stock Gainers ---
   Symbol                                 Name
0    PRVA            Privia Health Group, Inc.
1    CELH               Celsius Holdings, Inc.
2    BTDR           Bitdeer Technologies Group
3     OLN                     Olin Corporation
4    BF-B             Brown-Forman Corporation
5     KGC             Kinross Gold Corporation
6      OS                      OneStream, Inc.
7     NGD                        New Gold Inc.
8    CNXC               Concentrix Corporation
9    DOOO                             BRP Inc.
10    SJM            The J. M. Smucker Company
11   AFRM                Affirm Holdings, Inc.
12    HCC      

## Part 2: Historical Data Retrieval

- Now need to call the API from Yahoo Finance and obtain historical data from it

In [22]:
# 1. Get the list of symbols from our scraped data
stock_symbols = gainers_df['Symbol'].tolist()

# 2. Download all data in a single, optimized batch request. Instead of sending one individual request, send one bulk
historical_data_raw = yf.download(
    tickers = stock_symbols,
    period = "1y",
    interval = "1mo",
    group_by = 'ticker', # Keeps data grouped by ticker, easier for processing
    auto_adjust = True,  # Automatically gets the adjusted close price
    threads = True       # Use multiple threads to speed up the download
)

# 3. Process the downloaded data
# The 'auto_adjust=True' parameter simplifies things. Now we just need to get the 'Close' column for each ticker.
if not historical_data_raw.empty:
    data_rows = []
    for symbol in stock_symbols:
        try:
            close_series = historical_data_raw[symbol]['Close']
            for date, value in close_series.items():
                data_rows.append({'Date': date, 'Symbol': symbol, 'Close': value})
        except KeyError:
            print(f"Data for symbol '{symbol}' was not found in the downloaded batch. Skipping.")

    historical_data = pd.DataFrame(data_rows)
    # 4. Clean the data: Drop any columns that are entirely empty (failed downloads)
    historical_data.dropna(axis=1, how='all', inplace=True)

    print("\nHistorical Adjusted Close Prices (12 Months):")
    print(historical_data.head())
else:
    print("The yfinance download returned an empty DataFrame. Check the stock symbols.")

[*********************100%***********************]  50 of 50 completed


Historical Adjusted Close Prices (12 Months):
        Date Symbol      Close
0 2024-09-01   PRVA  18.209999
1 2024-10-01   PRVA  18.360001
2 2024-11-01   PRVA  21.480000
3 2024-12-01   PRVA  19.549999
4 2025-01-01   PRVA  22.850000


## Part 3: Portfolio Construction & Analysis


In [31]:
import numpy as np 

# Have a more analysis-friendly format by pivoting the DataFrame
prices_df = historical_data.pivot(index='Date', columns='Symbol', values='Close')
# Ensure the data is sorted chronologically
prices_df.sort_index(inplace=True)
prices_df_clean=prices_df.copy()
prices_df_clean = prices_df_clean.iloc[:-1] #No price at the end of August yet
#For the analysis , we need stocks with complete data
prices_df_clean = prices_df_clean.dropna(axis=1, how='any')


# Portfolio Construction & Analysis

# First 6 Months: Need 7 prices points
first_period_prices = prices_df_clean.iloc[:7]
# Calculate the monthly returns for this period.
first_period_returns = first_period_prices.pct_change()

# Calculate the cumulative return for each stock over these 6 months.
cumulative_returns = (1 + first_period_returns).prod() - 1

# Select the top 10 performing stocks based on this metric.
top_10_stocks = cumulative_returns.nlargest(10)
portfolio_symbols = top_10_stocks.index.tolist()

print("\n Portfolio Selection")
print("Selection Strategy: Top 10 stocks by cumulative return in the first 6 months.")
print(top_10_stocks)




 Portfolio Selection
Selection Strategy: Top 10 stocks by cumulative return in the first 6 months.
Symbol
GH      0.857018
SSRM    0.765845
SOUN    0.742489
HMY     0.459887
GFI     0.456659
AEM     0.357891
KGC     0.351284
FSM     0.317495
NGD     0.288194
WPM     0.274093
dtype: float64


In [39]:
# Calculate portfolio performance over the last 5 months (because we are missing 1 data point)

# We select only the columns for our 10 chosen portfolio stocks.
second_period_prices = prices_df_clean.loc[prices_df_clean.index[6:], portfolio_symbols]

# 1. Individual Stock Monthly Returns
# Calculate the monthly percentage returns for our 10 selected stocks.
individual_returns_last_5m = second_period_prices.pct_change()

# 2. Portfolio Monthly Returns
# Assumption: An equal-weighted portfolio (10% in each of the 10 stocks).
portfolio_monthly_returns = individual_returns_last_5m.mean(axis=1)

print("\nEqual-Weighted Portfolio Monthly Returns:")
print(portfolio_monthly_returns)

# 3. Cumulative Portfolio Return over the 5 months
cumulative_portfolio_return = (1 + portfolio_monthly_returns).prod() - 1
print(f"\nCumulative Portfolio Return over the last 5 months: {cumulative_portfolio_return:.2%}")


Equal-Weighted Portfolio Monthly Returns:
Date
2025-03-01         NaN
2025-04-01    0.086141
2025-05-01    0.010221
2025-06-01    0.074670
2025-07-01   -0.039810
2025-08-01    0.304939
dtype: float64

Cumulative Portfolio Return over the last 5 months: 47.75%


# Exercise 2: Reddit API Data Collection

## Part 1: Reddit API Setup & Data Collection

In [41]:
!pip install praw

  Using cached praw-7.8.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached prawcore-2.4.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached update_checker-0.18.0-py3-none-any.whl.metadata (2.3 kB)
Using cached praw-7.8.1-py3-none-any.whl (189 kB)
Using cached prawcore-2.4.0-py3-none-any.whl (17 kB)
Using cached update_checker-0.18.0-py3-none-any.whl (7.0 kB)

   -------------------------- ------------- 2/3 [praw]
   ---------------------------------------- 3/3 [praw]



In [45]:
# Import necessary libraries
import praw
import pandas as pd
from dotenv import load_dotenv
import os

# -- Environment Setup --
# Load environment variables from a .env file
load_dotenv()

# --- API Connection (PRAW) ---
# Retrieve credentials from environment variables
CLIENT_ID = os.getenv("REDDIT_CLIENT_ID")
CLIENT_SECRET = os.getenv("REDDIT_CLIENT_SECRET")
USER_AGENT = os.getenv("REDDIT_USER_AGENT")


# Initialize the Reddit instance
reddit = praw.Reddit(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    user_agent=USER_AGENT,
)

print("Successfully connected to Reddit API:", reddit.user.me())

Successfully connected to Reddit API: None


## Part 2: Collect Data and Storage

In [47]:
# Collect Posts from Subreddits
subreddits_to_scan = ['politics', 'PoliticalDiscussion', 'worldnews']
posts_data = []

for sub_name in subreddits_to_scan:
    subreddit = reddit.subreddit(sub_name)
    # Get the top 20 hot posts
    for post in subreddit.hot(limit=20):
        posts_data.append({
            'subreddit': sub_name,
            'id': post.id,
            'title': post.title,
            'score': post.score,
            'num_comments': post.num_comments,
            'url': post.url
        })

# Store post data in a DataFrame
posts_df = pd.DataFrame(posts_data)

# Collect Comments
comments_data = []

# Iterate through the collected posts to get comments
for post_id in posts_df['id']:
    submission = reddit.submission(id=post_id)
    # Using .comments.list() might include "MoreComments" objects, so we handle that
    submission.comments.replace_more(limit=0) # Remove "MoreComments" objects
    comment_count = 0
    for comment in submission.comments.list():
        if comment_count < 5:
             comments_data.append({
                'post_id': post_id,
                'body': comment.body,
                'score': comment.score
            })
             comment_count += 1
        else:
            break

# Store comment data in a DataFrame
comments_df = pd.DataFrame(comments_data)

# Storage: Linking comments to posts
# We can merge the two DataFrames to have a comprehensive dataset
merged_df = pd.merge(posts_df, comments_df, left_on='id', right_on='post_id')
print("\nMerged Posts and Comments Data:")
print(merged_df.head())


Merged Posts and Comments Data:
  subreddit       id                                              title  \
0  politics  1n62egw  Donald Trump Declares D.C. a 'Crime Free Zone'...   
1  politics  1n62egw  Donald Trump Declares D.C. a 'Crime Free Zone'...   
2  politics  1n62egw  Donald Trump Declares D.C. a 'Crime Free Zone'...   
3  politics  1n62egw  Donald Trump Declares D.C. a 'Crime Free Zone'...   
4  politics  1n62egw  Donald Trump Declares D.C. a 'Crime Free Zone'...   

   score_x  num_comments                                                url  \
0     8368           474  https://www.rollingstone.com/politics/politics...   
1     8368           474  https://www.rollingstone.com/politics/politics...   
2     8368           474  https://www.rollingstone.com/politics/politics...   
3     8368           474  https://www.rollingstone.com/politics/politics...   
4     8368           474  https://www.rollingstone.com/politics/politics...   

   post_id                               